# Step 19 — Where does a pretrained model commit? All 44 CIFAR-10 pairs, against a held-out reference

**Principle.** Run with the exact score, the reverse SDE reproduces the forward process's *joint* law over noise levels. So, for any order parameter $\phi$:

- *Model side:* run the model's reverse SDE, snapshot each trajectory at every $\sigma$, weight the trajectory by the classifier's terminal posterior for classes $a$ and $b$, and take the distribution of the pair log-odds $\phi_{ab}(x_\sigma)$.
- *Reference side:* forward-noise real images, weight them by the same classifier at the same terminal noise level ($\sigma_{\min}=0.05$), and take the same distribution.

With an exact score the two coincide. Any difference is therefore the model's. This needs no divergence term and no Hutchinson probes, and one sampling run serves every pair at every $\sigma$. The deterministic ODE preserves marginals but not the joint law, so it does *not* have this property. Part A demonstrates both facts on an exactly solvable 3-class mixture.

**Parts.**
- **A.** Control validation (CPU).
- **B.** Reference logits on the CIFAR-10 test set (the population reference) and the train set.
- **C.** Pretrained EDM (unconditional, 56M parameters): reverse SDE, with $N$ sized automatically to a time budget. A smaller deterministic batch is also run, for commitment only.
- **D.** For each of the 44 pairs: $\sigma_{\rm bif}$ of model vs reference with bootstrap intervals, $h$ of model vs reference, and commitment times. The commitment time is defined by magnitude: the last noise level after which the standardised log-odds stays beyond $\pm0.5$ on its final side.

**Runtime.** Self-contained. GPU on, Internet on. Run it as a committed version (Save & Run All); budget ≈ 6–8 h, most of it part C. If the Step 16 or 18 output is mounted, the classifier is loaded instead of retrained.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import sys, subprocess, pickle, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); torch.manual_seed(0); plt.rcParams['figure.dpi'] = 120
TIME_BUDGET_SDE_H = 4.0          # hours for the model's reverse-SDE generation; N is sized to fit
SIGMA_MAX, SIGMA_MIN, RHO, STEPS = 80.0, 0.05, 7.0, 256
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}
PAIRS = list(combinations(range(10), 2))

# ---------- weighted KDE, stationary points, transition scan, cusp fit (shared by all parts) ----------
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    """Weighted KDE score with Silverman bandwidth. Model and reference are always compared at EQUAL sample size (see match_n),
    so the bandwidth rule and the detection noise are identical on both sides under the null of an exact score."""
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_fit(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def scan_transition(sigmas_desc, dist_at):
    """sigmas_desc: noise grid in DESCENDING order. dist_at(i) -> (sm, p) or None if unresolvable (declared unimodal).
    Returns (sigma_bif, jump, index) = first grid point, scanning downward, that is bimodal after at least one unimodal point."""
    prev_max = None
    for i, s in enumerate(sigmas_desc):
        d = dist_at(i)
        pts = [(0.0, 'max')] if d is None else stationary(*d)
        if n_max(pts) >= 2 and prev_max is not None:
            mx = sorted(p for p, k in pts if k == 'max'); mn = [p for p, k in pts if k == 'min']
            return float(s), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev_max)), i
        if n_max(pts) == 1: prev_max = [p for p, k in pts if k == 'max'][0]
    return None
def edm_sched(steps=STEPS, s_max=SIGMA_MAX, s_min=SIGMA_MIN, rho=RHO):
    i = np.arange(steps); return (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
SCHED = edm_sched(); REC = np.where((SCHED >= 0.08) & (SCHED <= 8.0))[0]
REC = REC[::max(1, len(REC)//60)]; SIG_REC = SCHED[REC]      # recorded noise levels, descending
print(f"schedule: {STEPS} steps; recording {len(REC)} noise levels between {SIG_REC.min():.3f} and {SIG_REC.max():.2f}")

## A. Validation on an exactly solvable 3-class mixture (CPU)

The reverse SDE with the exact score should reproduce the reference transition. The ODE, weighted by terminal class, need not, because it preserves marginals but not the joint law. Commitment times are measured on both samplers.

In [ ]:
def make_mix3(d=256, widths=(0.15, 0.45, 0.3), iso=0.3, weights=(0.34, 0.33, 0.33), seed=0, scale=0.25):
    """3-class anisotropic Gaussian mixture; 'scale' shrinks the geometry so the a/b transition falls mid-grid (sigma ~ 1-2)."""
    r = np.random.default_rng(seed); M = np.sign(r.standard_normal((3, d))); M[2] *= 0.0; M[2][:d//2] = np.sign(r.standard_normal(d//2)); lw = np.log(np.array(weights))
    M = scale*M; widths = tuple(scale*w for w in widths); iso = scale*iso
    U = M/np.linalg.norm(M, axis=1, keepdims=True)
    def comps(x, sigma):
        a = iso**2 + sigma**2; L = []; Gd = []
        for c in range(3):
            b = widths[c]**2*d; dx = x - M[c]; proj = dx @ U[c]
            L.append(lw[c] - 0.5*(((dx**2).sum(1) - (b/(a+b))*proj**2)/a) - 0.5*((d-1)*np.log(a) + np.log(a+b))); Gd.append(-(dx - (b/(a+b))*proj[:, None]*U[c][None, :])/a)
        L = np.array(L); P = np.exp(L - L.max(0)); P /= P.sum(0); return L, P, Gd
    def score(x, sigma): L, P, Gd = comps(x, sigma); return sum(P[c][:, None]*Gd[c] for c in range(3))
    def logits(x, sigma): return comps(x, sigma)[0].T                   # exact class log-joint = optimal noise-conditional classifier
    def sample0(n, r):
        c = r.choice(3, size=n, p=np.array(weights)); x = np.empty((n, d))
        for k in range(3):
            j = c == k; x[j] = M[k] + widths[k]*np.sqrt(d)*r.standard_normal((j.sum(), 1))*U[k] + iso*r.standard_normal((j.sum(), d))
        return x
    return score, logits, sample0, d
score, logits, sample0, dC = make_mix3(); rC = np.random.default_rng(1); pa, pb = 0, 1
def softmax(L): e = np.exp(L - L.max(1, keepdims=True)); return e/e.sum(1, keepdims=True)
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
# reference: exact forward joint  x_min = x0 + s_min e1 ; x_s = x_min + sqrt(s^2 - s_min^2) e2
n = 40000; x0 = sample0(n, rC); xmin = x0 + SIGMA_MIN*rC.standard_normal(x0.shape); Wf = softmax(logits(xmin, SIGMA_MIN)); ZR = []; STD = []
for s in SIG_REC:
    xs = xmin + np.sqrt(s**2 - SIGMA_MIN**2)*rC.standard_normal(x0.shape); L = logits(xs, s); z = L[:, pa]-L[:, pb]; ZR.append(z); STD.append(std_stats(z, Wf[:, pa], Wf[:, pb]))
def gen(n, kind, steps=STEPS):
    sch = edm_sched(steps); rec = [int(np.argmin(np.abs(sch - s))) for s in SIG_REC]; x = rC.standard_normal((n, dC))*sch[0]; Z = {}
    for i in range(steps):
        s0 = sch[i]; s1 = sch[i+1] if i+1 < steps else SIGMA_MIN
        if i in rec: L = logits(x, s0); Z[i] = L[:, pa]-L[:, pb]
        if s1 >= s0: continue
        sc = score(x, s0)
        if kind == 'sde': x = x + (s1-s0)*(-2*s0*sc) + np.sqrt(2*s0*(s0-s1))*rC.standard_normal(x.shape)
        else:
            d0 = -s0*sc; x1 = x + (s1-s0)*d0; d1 = -s1*score(x1, s1); x = x + (s1-s0)*0.5*(d0+d1)
    W = softmax(logits(x, SIGMA_MIN)); return np.array([Z[i] for i in rec]), W
def transition_from(Zlist, W, STD, idx=None):
    def dist(i):
        mid, half, sep = STD[i]
        if sep < 0.3: return None
        z = Zlist[i] if idx is None else Zlist[i][idx]; ww = W if idx is None else W[idx]
        return wkde_score((z-mid)/half, ww[:, pa]+ww[:, pb])
    return scan_transition(SIG_REC, dist)
def commit_sigma(Zlist, W, STD, thr=0.5):
    """Per trajectory ending in a or b: the largest recorded sigma after which the standardised log-odds stays beyond +/-thr on its final side."""
    lab = W.argmax(1); keep = np.isin(lab, [pa, pb]); fin = np.where(lab[keep] == pa, 1.0, -1.0)
    M_ = np.array([(Zlist[i][keep]-STD[i][0])/STD[i][1] for i in range(len(SIG_REC))])        # (grid desc, traj)
    ok = (np.sign(M_) == fin[None, :]) & (np.abs(M_) >= thr); tail = np.flip(np.cumprod(np.flip(ok, 0), 0), 0).astype(bool)   # tail[i] = ok for all j >= i
    first = np.where(tail.any(0), tail.argmax(0), -1); return np.where(first >= 0, SIG_REC[np.maximum(first, 0)], np.nan)
t0 = time.time(); NTRAJ = 20000
for kind, steps in [('sde', 256), ('sde', 512), ('ode', 64)]:
    Z, W = gen(NTRAJ, kind, steps); Z = list(Z); cs = commit_sigma(Z, W, STD); rat = []
    for rep in range(10):                                       # equal-size resamples of both sides
        im = rC.integers(NTRAJ, size=NTRAJ); ir = rC.choice(len(Wf), size=NTRAJ, replace=False)
        tm = transition_from(Z, W, STD, im); tr = transition_from(ZR, Wf, STD, ir)
        if tm and tr: rat.append(tm[0]/tr[0])
    rat = np.array(rat); print(f"{kind} {steps:>3} steps: sigma_bif ratio model/reference at equal n: median {np.median(rat):.3f} [{rat.min():.3f}, {rat.max():.3f}] over {len(rat)} resamples;  commit sigma median {np.nanmedian(cs):.3f} [IQR {np.nanpercentile(cs,25):.3f}, {np.nanpercentile(cs,75):.3f}]  ({time.time()-t0:.0f}s)")
print("Pass criterion: SDE median ratio near 1 with a spread that brackets 1 (both step counts). The ODE may deviate: terminal-class weighting under the ODE does not reproduce the forward joint law.")

## B. CIFAR-10: classifier, and reference logits on the test and train sets

In [ ]:
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True); dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True)
XTR = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YTR = np.asarray(ds.targets)
XTE = torch.from_numpy((np.asarray(dst.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YTE = np.asarray(dst.targets)
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
def train_clf10(seed=0, epochs=25, bs=256, s_lo=0.05, s_hi=10.0):
    torch.manual_seed(seed); Xd = XTR.to(dev); Yd = torch.from_numpy(YTR).long().to(dev); clf = NoisyResNet().to(dev); opt = torch.optim.AdamW(clf.parameters(), 2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs*(len(Xd)//bs)); t0 = time.time()
    for ep in range(epochs):
        clf.train(); perm = torch.randperm(len(Xd), device=dev)
        for i in range(0, len(Xd)-bs+1, bs):
            j = perm[i:i+bs]; x0 = Xd[j]
            if torch.rand(1).item() < 0.5: x0 = x0.flip(3)
            sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(s_lo), math.log(s_hi))); xt = x0 + sigma.view(-1, 1, 1, 1)*torch.randn_like(x0)
            loss = F.cross_entropy(clf(xt, sigma), Yd[j]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    clf.eval(); torch.save(clf.state_dict(), '/kaggle/working/noisy_resnet_seed0.pt'); print(f"classifier trained ({time.time()-t0:.0f}s)"); del Xd; torch.cuda.empty_cache(); return clf
found = glob.glob('/kaggle/input/**/noisy_resnet_seed0.pt', recursive=True)
if found: CLF = NoisyResNet().to(dev); CLF.load_state_dict(torch.load(found[0], map_location=dev)); CLF.eval(); print('loaded classifier from', found[0])
else: CLF = train_clf10(0)
@torch.no_grad()
def clf_logits(x, sigma, bs=1000):
    out = []
    for xb in x.split(bs):
        with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')): out.append(CLF(xb, torch.full((len(xb),), float(sigma), device=dev)).float())
    return torch.cat(out)
@torch.no_grad()
def reference_logits(Xsrc, repeats, seed):
    """Exact forward joint: x_min = x0 + s_min e1 (terminal weights), x_s = x_min + sqrt(s^2 - s_min^2) e2 at each recorded sigma. Returns (L[g, n, 10] float16, W[n, 10])."""
    g = torch.Generator(device=dev).manual_seed(seed); Ls = np.zeros((len(SIG_REC), len(Xsrc)*repeats, 10), np.float16); Ws = []
    for r_ in range(repeats):
        for i0 in range(0, len(Xsrc), 2000):
            x0 = Xsrc[i0:i0+2000].to(dev); xmin = x0 + SIGMA_MIN*torch.randn(x0.shape, generator=g, device=dev); Ws.append(F.softmax(clf_logits(xmin, SIGMA_MIN), 1).cpu().numpy())
            off = r_*len(Xsrc) + i0
            for gi, s in enumerate(SIG_REC):
                xs = xmin + math.sqrt(s**2 - SIGMA_MIN**2)*torch.randn(x0.shape, generator=g, device=dev); Ls[gi, off:off+len(x0)] = clf_logits(xs, s).cpu().numpy().astype(np.float16)
    return Ls, np.concatenate(Ws)
t0 = time.time(); L_TE, W_TE = reference_logits(XTE, 4, 11); print(f"test reference: {L_TE.shape} ({time.time()-t0:.0f}s)")
t0 = time.time(); L_TR, W_TR = reference_logits(XTR, 1, 12); print(f"train reference: {L_TR.shape} ({time.time()-t0:.0f}s)")
print('classifier terminal accuracy on test (argmax of terminal weights vs label):', float((W_TE[:len(YTE)].argmax(1) == YTE).mean()))

## C. Pretrained EDM: reverse-SDE trajectories (N sized to the time budget), plus a deterministic batch for commitment

In [ ]:
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: NET = pickle.load(f)['ema'].to(dev).eval()
@torch.no_grad()
def D(x, s): return NET(x, torch.full((len(x),), float(s), device=dev)).float()
xb = torch.randn(64, 3, 32, 32, device=dev)*2.0; d32 = D(xb, 2.0)
if hasattr(NET, 'use_fp16'):
    NET.use_fp16 = True; d16 = D(xb, 2.0); rel = float((d16-d32).norm()/d32.norm())
    if rel > 1e-2: NET.use_fp16 = False
    print(f"fp16 relative deviation {rel:.2e}; use_fp16={NET.use_fp16}")
@torch.no_grad()
def generate(n_chunk, kind='sde', steps=STEPS, seed=0):
    g = torch.Generator(device=dev).manual_seed(seed); sch = edm_sched(steps); rec = {int(np.argmin(np.abs(sch - s))): gi for gi, s in enumerate(SIG_REC)}
    x = torch.randn(n_chunk, 3, 32, 32, generator=g, device=dev)*sch[0]; Lrec = np.zeros((len(SIG_REC), n_chunk, 10), np.float16)
    for i in range(steps):
        s0 = float(sch[i]); s1 = float(sch[i+1]) if i+1 < steps else SIGMA_MIN
        if i in rec: Lrec[rec[i]] = clf_logits(x, s0).cpu().numpy().astype(np.float16)
        if s1 >= s0: continue
        if kind == 'sde':
            sc = (D(x, s0) - x)/s0**2; x = x + (s1-s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0-s1))*torch.randn(x.shape, generator=g, device=dev)
        else:
            d0 = (x - D(x, s0))/s0; x1 = x + (s1-s0)*d0; d1 = (x1 - D(x1, s1))/s1; x = x + (s1-s0)*0.5*(d0+d1)
    return Lrec, F.softmax(clf_logits(x, SIGMA_MIN), 1).cpu().numpy()
CHUNK = 500; t0 = time.time(); Lm, Wm = generate(CHUNK, 'sde', seed=100); dt = time.time()-t0
n_chunks = max(8, int(TIME_BUDGET_SDE_H*3600/dt)); print(f"one chunk of {CHUNK} SDE trajectories: {dt:.0f}s -> {n_chunks} chunks ({n_chunks*CHUNK} trajectories) within {TIME_BUDGET_SDE_H} h")
LM = [Lm]; WM = [Wm]
for c in range(1, n_chunks):
    Lm, Wm = generate(CHUNK, 'sde', seed=100+c); LM.append(Lm); WM.append(Wm)
    if c % 10 == 0: print(f"  {c+1}/{n_chunks} chunks ({time.time()-t0:.0f}s)")
L_M = np.concatenate(LM, 1); W_M = np.concatenate(WM); del LM, WM
freq = np.bincount(W_M.argmax(1), minlength=10)/len(W_M); print('generated class frequencies (SDE):', {CLASSES[c]: round(float(freq[c]), 3) for c in range(10)})
t0 = time.time(); LO = []; WO = []
for c in range(6):
    Lo, Wo = generate(CHUNK, 'ode', steps=64, seed=900+c); LO.append(Lo); WO.append(Wo)
L_O = np.concatenate(LO, 1); W_O = np.concatenate(WO); print(f"deterministic batch: {len(W_O)} trajectories ({time.time()-t0:.0f}s)")
np.savez_compressed('/kaggle/working/step19_logits.npz', L_TE=L_TE, W_TE=W_TE, L_TR=L_TR, W_TR=W_TR, L_M=L_M, W_M=W_M, L_O=L_O, W_O=W_O, SIG_REC=SIG_REC)

## D. Model vs reference on all pairs: transition, asymmetry, commitment

In [ ]:
def pair_std(L, W, a, b):
    """(mid, half, sep) per recorded sigma from the REFERENCE's weighted class-conditional means of the pair log-odds."""
    out = []
    for gi in range(len(SIG_REC)):
        z = L[gi, :, a].astype(np.float64) - L[gi, :, b].astype(np.float64); out.append(std_stats(z, W[:, a] + 1e-12, W[:, b] + 1e-12))
    return out
def pair_transition(L, W, a, b, STD_, idx=None):
    W_ = W if idx is None else W[idx]
    def dist(gi):
        mid, half, sep = STD_[gi]
        if sep < 0.3: return None
        z = L[gi, :, a].astype(np.float64) - L[gi, :, b].astype(np.float64); z = z if idx is None else z[idx]
        return wkde_score((z - mid)/half, W_[:, a] + W_[:, b])
    tr = scan_transition(SIG_REC, dist)
    if tr is None: return None
    sm, p = dist(tr[2]); return dict(sigma=tr[0], jump=tr[1], **cusp_fit(sm, p))
def pair_commit(L, W, a, b, STD_, thr=0.5):
    lab = W.argmax(1); keep = np.isin(lab, [a, b]); fin = np.where(lab[keep] == a, 1.0, -1.0)
    M_ = np.array([((L[gi, keep, a].astype(np.float64) - L[gi, keep, b]) - STD_[gi][0])/STD_[gi][1] for gi in range(len(SIG_REC))])
    ok = (np.sign(M_) == fin[None, :]) & (np.abs(M_) >= thr); tail = np.flip(np.cumprod(np.flip(ok, 0), 0), 0).astype(bool)
    first = np.where(tail.any(0), tail.argmax(0), -1); cs = np.where(first >= 0, SIG_REC[np.maximum(first, 0)], np.nan); return np.nanmedian(cs), np.nanpercentile(cs, 25), np.nanpercentile(cs, 75), keep.sum()
rngb = np.random.default_rng(5); NB = 30; rows = []; t0 = time.time()
print(f"{'pair':>20} {'ref(test)':>9} {'ref(train)':>10} {'model SDE':>10} {'ratio [95% CI]':>22} {'h ref':>7} {'h model':>8} {'commit SDE':>11} {'commit ODE':>11} {'n traj':>7}")
NM = len(W_M); print(f"model trajectories: {NM}; reference subsampled to the same size in every comparison")
for a, b in PAIRS:
    STD_ = pair_std(L_TE, W_TE, a, b)
    ir0 = rngb.choice(len(W_TE), size=min(NM, len(W_TE)), replace=False); rt = pair_transition(L_TE, W_TE, a, b, STD_, ir0)
    it0 = rngb.choice(len(W_TR), size=min(NM, len(W_TR)), replace=False); rtr = pair_transition(L_TR, W_TR, a, b, pair_std(L_TR, W_TR, a, b), it0); rm = pair_transition(L_M, W_M, a, b, STD_)
    name = CLASSES[a] + '/' + CLASSES[b]
    if rt is None or rm is None: print(f"{name:>20}  reference or model not bimodal (ref {rt is not None}, model {rm is not None})"); continue
    bm = [pair_transition(L_M, W_M, a, b, STD_, rngb.integers(NM, size=NM)) for _ in range(NB)]; br = [pair_transition(L_TE, W_TE, a, b, STD_, rngb.choice(len(W_TE), size=min(NM, len(W_TE)), replace=True)) for _ in range(NB)]
    ratios = np.array([x['sigma']/y['sigma'] for x, y in zip(bm, br) if x and y]); lo, hi = (np.percentile(ratios, [2.5, 97.5]) if len(ratios) > 10 else (np.nan, np.nan))
    cm = pair_commit(L_M, W_M, a, b, STD_); co = pair_commit(L_O, W_O, a, b, STD_)
    rows.append(dict(pair=(a, b), ref=rt['sigma'], ref_tr=(rtr['sigma'] if rtr else np.nan), model=rm['sigma'], lo=lo, hi=hi, h_ref=rt['h'], h_model=rm['h'], commit_sde=cm[0], commit_ode=co[0], ntraj=cm[3]))
    print(f"{name:>20} {rt['sigma']:9.3f} {(rtr['sigma'] if rtr else float('nan')):10.3f} {rm['sigma']:10.3f} {rm['sigma']/rt['sigma']:8.2f} [{lo:5.2f},{hi:5.2f}] {rt['h']:+7.3f} {rm['h']:+8.3f} {cm[0]:11.3f} {co[0]:11.3f} {cm[3]:7d}")
print(f"({time.time()-t0:.0f}s)")
R = np.array([r['model']/r['ref'] for r in rows]); LO_ = np.array([r['lo'] for r in rows]); HI_ = np.array([r['hi'] for r in rows])
sr = np.array([r['ref'] for r in rows]); sm_ = np.array([r['model'] for r in rows]); hr = np.array([r['h_ref'] for r in rows]); hm = np.array([r['h_model'] for r in rows])
cs = np.array([r['commit_sde'] for r in rows]); co = np.array([r['commit_ode'] for r in rows])
print(f"\n{len(rows)} pairs resolved on both sides. model/ref sigma_bif: median {np.median(R):.2f} (IQR {np.percentile(R,25):.2f}-{np.percentile(R,75):.2f}); "
      f"pairs with CI entirely above 1: {(LO_ > 1).sum()}, entirely below 1: {(HI_ < 1).sum()}, overlapping 1: {((LO_ <= 1) & (HI_ >= 1)).sum()}")
print(f"corr(log sigma_bif model, ref) = {np.corrcoef(np.log(sm_), np.log(sr))[0,1]:.3f};  corr(h model, h ref) = {np.corrcoef(hm, hr)[0,1]:.3f}; sign agreement {np.mean(np.sign(hm) == np.sign(hr)):.2f}")
kind = np.array(['va' if ((p['pair'][0] in VEH) != (p['pair'][1] in VEH)) else 'vv' if p['pair'][0] in VEH else 'aa' for p in rows])
for k in ['va', 'vv', 'aa']: print(f"  {k}: median sigma_bif ref {np.median(sr[kind == k]):.2f}, model {np.median(sm_[kind == k]):.2f}, median ratio {np.median(R[kind == k]):.2f};  median commit SDE {np.nanmedian(cs[kind == k]):.2f}, ODE {np.nanmedian(co[kind == k]):.2f}")
print(f"commitment lag: median sigma_bif(ref)/commit(SDE) = {np.nanmedian(sr/cs):.2f}; median sigma_bif(model)/commit(SDE) = {np.nanmedian(sm_/cs):.2f};  ODE commit / SDE commit = {np.nanmedian(co/cs):.2f}")
fig, ax = plt.subplots(1, 3, figsize=(14, 4)); col = ['C0' if k == 'va' else 'C1' if k == 'vv' else 'C2' for k in kind]
ax[0].errorbar(sr, sm_, yerr=[sm_ - LO_*sr, HI_*sr - sm_], fmt='none', ecolor='grey', lw=0.6); ax[0].scatter(sr, sm_, s=16, c=col); lim = [min(sr.min(), sm_.min())*0.8, max(sr.max(), sm_.max())*1.2]; ax[0].plot(lim, lim, 'k--', lw=0.7)
ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].set_xlabel('sigma_bif, reference (test set)'); ax[0].set_ylabel('sigma_bif, pretrained EDM (reverse SDE)'); ax[0].set_title('where the model commits vs the data')
ax[1].scatter(hr, hm, s=16, c=col); ax[1].axline((0, 0), slope=1, color='k', lw=0.7); ax[1].set_xlabel('h, reference'); ax[1].set_ylabel('h, model'); ax[1].set_title('asymmetry')
ax[2].scatter(sr, cs, s=16, c=col, label='SDE'); ax[2].scatter(sr, co, s=16, marker='x', c=col, label='ODE'); ax[2].plot(lim, lim, 'k--', lw=0.7); ax[2].set_xscale('log'); ax[2].set_yscale('log'); ax[2].set_xlabel('sigma_bif, reference'); ax[2].set_ylabel('commitment sigma (median)'); ax[2].legend(fontsize=7); ax[2].set_title('when trajectories commit')
plt.tight_layout(); plt.show()

## What to report back
- **A.** The three control lines. The SDE ratio must be close to 1.
- **B.** The classifier's terminal accuracy line.
- **C.** The trajectory count, and the generated class frequencies.
- **D.** The per-pair table, the summary lines and the three panels. The paper's remaining claims turn on these numbers:
  - the model/reference ratio, with how many pairs have an interval entirely above or below 1;
  - $h$ agreement between model and reference;
  - whether the vehicle/animal hierarchy is present in the model;
  - the commitment lag.